# Plant Disease Detection using a Transformer

Interactive mirror of `src/train.py`. Trains a **Vision Transformer** on the PlantVillage leaf dataset.

**Same data as the CNN project.** The images, the stratified train/val/test split CSVs and the
diagnosis knowledge base are read (read-only) from the sibling `../Plant-Disease-Detection-CNN`
folder — nothing is duplicated and nothing there is modified. The split is verified (disjoint,
exhaustive, all files present) and fingerprinted, so the CNN-vs-transformer comparison is honest.

> A full run takes ~8 min/epoch on CPU (~4.5 h for 32 epochs). The last cell reuses the
> checkpoint already in `models/` for a quick end-to-end demonstration without retraining.

In [ ]:
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, confusion_matrix

PROJECT_ROOT = os.path.abspath(os.path.join('..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

import train as T
from model import build_model, count_parameters
from preprocessing import build_loaders, load_class_indices, load_splits

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 96
EPOCHS = 32
BATCH_SIZE = 64

torch.set_num_threads(12)
print('torch', torch.__version__, '| device', DEVICE, '| threads', torch.get_num_threads())

## 1. Load and verify the shared split

`load_splits()` resolves the CSVs, asserts the three splits are disjoint and exhaustive, checks
that every listed image exists, and returns a SHA-256 fingerprint per split.

In [ ]:
frames, provenance = load_splits(verbose=True)
class_indices = load_class_indices(provenance['split_dir'])
NUM_CLASSES = provenance['num_classes']
CLASS_NAMES = [class_indices[i] for i in range(NUM_CLASSES)]

print('source :', provenance['path_root'])
print('counts :', provenance['counts'])
print('disjoint / all present:', provenance['splits_disjoint'], '/', provenance['all_images_exist'])

counts = pd.DataFrame({
    'train': provenance['class_distribution']['train'],
    'val': provenance['class_distribution']['val'],
    'test': provenance['class_distribution']['test'],
}).fillna(0).astype(int)
counts.plot(kind='barh', figsize=(8, 5), title='Shared split - images per class')
plt.tight_layout()

## 2. Data loaders

Augmentation mirrors the CNN's intent (flip / rotate / zoom / translate) using the PyTorch
equivalent stack, plus RandAugment and RandomErasing which the from-scratch transformer needs.
The **evaluation** transform is deliberately identical to the CNN's preprocessing: plain resize
to 96x96 and `/255` scaling.

In [ ]:
train_loader, val_loader, test_loader = build_loaders(
    frames, img_size=IMG_SIZE, batch_size=BATCH_SIZE, workers=4, seed=42)

xb, yb = next(iter(train_loader))
print('train batch:', tuple(xb.shape), xb.dtype, f'range [{xb.min():.2f}, {xb.max():.2f}]')
print('steps/epoch:', len(train_loader))
print('torch.max_threads:', torch.get_num_threads())

## 3. The Vision Transformer

Patch embedding (8x8 stride) -> 144 tokens + [CLS] -> 12 pre-norm self-attention blocks ->
linear head. No convolution in the feature extractor.

In [ ]:
model = build_model(arch='vit_tiny', img_size=IMG_SIZE, patch_size=8,
                  num_classes=NUM_CLASSES, drop_path_rate=0.05).to(DEVICE)
print(f'params {count_parameters(model):,} | tokens {model.num_patches + 1} | dim {model.embed_dim} | depth {model.depth}')
model

## 4. Train

AdamW (no weight decay on bias/norm/pos/CLS), 3-epoch warmup + cosine decay, Mixup/CutMix with
soft targets, label smoothing, EMA. Raw and EMA weights are both validated each epoch and the
better one is kept. Set `EPOCHS = 0` to skip and evaluate the shipped checkpoint instead.

In [ ]:
T.set_seeds(42)
use_amp = T.cpu_bf16_supported()
optimizer = T.build_optimizer(model, lr=1e-3, weight_decay=0.05)
steps_per_epoch = len(train_loader)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, T.lr_lambda_factory(steps_per_epoch * EPOCHS, steps_per_epoch * 3, 0.01))
ema = T.ModelEMA(model, decay=0.9998)
criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
best = {'val_accuracy': -1.0}
history = []

if EPOCHS > 0:
    import time
    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        model.train()
        run_loss, seen = 0.0, 0
        for step, (x, y) in enumerate(train_loader):
            x, y = x.to(DEVICE), y.to(DEVICE)
            x, soft, _ = T.apply_mixup_cutmix(x, y, NUM_CLASSES, 0.2, 1.0, 0.5, 0.1)
            with T.amp_context(use_amp):
                loss = T.soft_target_ce(model(x), soft).float()
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            ema.update(model, epoch * steps_per_epoch + step)
            run_loss += loss.item() * y.size(0)
            seen += y.size(0)
            if step % 50 == 0:
                print(f'  epoch {epoch:3d} step {step + 1:4d}/{steps_per_epoch} loss {loss.item():.4f}', flush=True)
        val = T.evaluate(model, val_loader, criterion, DEVICE, NUM_CLASSES, use_amp=use_amp)
        ema_val = T.evaluate(ema.module, val_loader, criterion, DEVICE, NUM_CLASSES, use_amp=use_amp)
        history.append({'epoch': epoch, 'train_loss': run_loss / seen,
                        'val_accuracy': val['accuracy'], 'ema_val_accuracy': ema_val['accuracy']})
        print(f"EPOCH {epoch:3d} | train {run_loss / seen:.4f} | val acc {val['accuracy']:.4f} "
              f"| ema val acc {ema_val['accuracy']:.4f} | {(time.time() - t0) / 60:.1f} min", flush=True)
        for src, m in (('raw', val), ('ema', ema_val)):
            if m['accuracy'] > best['val_accuracy']:
                best = {'val_accuracy': m['accuracy'], 'epoch': epoch, 'source': src}
                state = model.state_dict() if src == 'raw' else ema.module.state_dict()
                torch.save(state, os.path.join(PROJECT_ROOT, 'models', 'vit_plant_disease.pt'))
    print('best:', best)
else:
    print('EPOCHS = 0 -> skipping training, using the checkpoint in models/')

## 5. Evaluate on the held-out test split

In [ ]:
final = build_model(arch='vit_tiny', img_size=IMG_SIZE, patch_size=8, num_classes=NUM_CLASSES,
                     drop_path_rate=0.05).to(DEVICE)
final.load_state_dict(torch.load(os.path.join(PROJECT_ROOT, 'models', 'vit_plant_disease.pt'),
                                map_location=DEVICE, weights_only=True))
test = T.evaluate(final, test_loader, criterion, DEVICE, NUM_CLASSES, return_probs=True, use_amp=use_amp)
print(f"TEST accuracy {test['accuracy']:.4f} | loss {test['loss']:.4f}\n")
print(classification_report(test['true'], test['pred'], target_names=CLASS_NAMES, digits=4, zero_division=0))

In [ ]:
cm = confusion_matrix(test['true'], test['pred'])
fig, ax = plt.subplots(figsize=(11, 10))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=90, fontsize=8)
ax.set_yticklabels(CLASS_NAMES, fontsize=8)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f"ViT-Tiny confusion matrix (test acc {test['accuracy']:.4f})")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=7)
fig.colorbar(im, ax=ax); fig.tight_layout()

## 6. Training curves

Train accuracy is measured on augmented, mixup-blended batches while validation is clean, so the
train metric is deliberately pessimistic - the gap is an augmentation artefact, not overfitting.

In [ ]:
hist_path = os.path.join(PROJECT_ROOT, 'reports', 'history.json')
if os.path.isfile(hist_path):
    h = json.load(open(hist_path))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(h['epoch'], h['train_loss'], label='train')
    axes[0].plot(h['epoch'], h['val_loss'], label='val')
    axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend()
    axes[1].plot(h['epoch'], h['val_accuracy'], label='val (raw)')
    axes[1].plot(h['epoch'], h['ema_val_accuracy'], label='val (EMA)')
    axes[1].axhline(0.8488, ls='--', c='gray', label='CNN project 0.8488')
    axes[1].set_title('Validation accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend()
    axes[2].plot(h['epoch'], np.array(h['epoch_seconds']) / 60)
    axes[2].set_title('Minutes per epoch'); axes[2].set_xlabel('epoch')
    fig.tight_layout()
else:
    print('no reports/history.json yet - run src/train.py first')

## 7. Sample predictions with the diagnosis layer

In [ ]:
from predict import PlantDiseasePredictor

predictor = PlantDiseasePredictor()
sample = frames['test'].sample(6, random_state=7)
for _, row in sample.iterrows():
    res = predictor.predict(row['path'])
    hit = res['top_class'] == row['label']
    d = res['diagnosis'] or {}
    print(f"{'OK ' if hit else 'MISS'} {res['top_confidence'] * 100:5.1f}%  {res['top_class']}")
    print(f"      crop={d.get('crop')} common_name={d.get('common_name')} confident={res['confident']}")

## 8. Non-leaf rejection

A blank image should be flagged low-confidence rather than silently mislabelled - that is what the
0.5 `confident` threshold is for.

In [ ]:
import io
from PIL import Image

buf = io.BytesIO()
Image.new('RGB', (256, 256), (255, 255, 255)).save(buf, format='JPEG')
buf.seek(0)
blank = predictor.predict(buf)
print(f"blank image -> {blank['top_class']} @ {blank['top_confidence'] * 100:.1f}% | confident={blank['confident']}")